In [1]:
from functools import reduce
import re

import pandas as pd
import numpy as np

from src.categories import subcat_to_cat

import pickle

In [2]:
# preds_path_ls = [
#     ('raw 0 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_0sh.bin'),
#     ('raw 5 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_5sh.bin'),
#     ('LoRA 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg13_0sh.bin'),
#     ('LoRA 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg13_5sh.bin'),
#     ('GSOFT 0 shot', r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg0_0sh.bin'),
#     ('GSOFT 5 shot', r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg0_5sh.bin'),
# ]

preds_path_ls = [
    ('raw 0 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_0sh.bin'),
    ('raw 5 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_5sh.bin'),
    ('LoRA 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg17_0sh.bin'),
    ('LoRA 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg17_5sh.bin'),
    ('LoRA-code 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg18-code-ift-MMLU_0sh.bin'),
    ('LoRA-code 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg18-code-ift-MMLU_5sh.bin'),
    ('GSOFT-code 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg3-code-ift-MMLU_0sh.bin'),
    ('GSOFT-code 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg3-code-ift-MMLU_5sh.bin'),
    # ('LoRA-CR 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg18-OOD-MMLU_0sh.bin'),
    # ('LoRA-CR 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg18-OOD-MMLU_5sh.bin'),
    # ('DoRA-CR 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\DoRA-cfg18-OOD-MMLU_0sh.bin'),
    # ('DoRA-CR 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\DoRA-cfg18-OOD-MMLU_5sh.bin'),
    # ('GSOFT-CR 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg3-OOD-MMLU_0sh.bin'),
    # ('GSOFT-CR 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg3-OOD-MMLU_5sh.bin'),
    # ('VeRA-CR 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\VeRA-cfg1-OOD-MMLU_0sh.bin'),
    # ('VeRA-CR 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\VeRA-cfg1-OOD-MMLU_5sh.bin'),
]

In [3]:
preds_dfs = {}

for run_name, preds_path in preds_path_ls:
    with open(preds_path, 'rb') as f:
        preds_dfs[run_name] = pickle.load(file=f)

        print(f"{run_name:16}: {preds_dfs[run_name].shape=}")

raw 0 shot      : preds_dfs[run_name].shape=(14042, 6)
raw 5 shot      : preds_dfs[run_name].shape=(14042, 6)
LoRA 0 shot     : preds_dfs[run_name].shape=(14042, 6)
LoRA 5 shot     : preds_dfs[run_name].shape=(14042, 6)
LoRA-code 0 shot: preds_dfs[run_name].shape=(14042, 7)
LoRA-code 5 shot: preds_dfs[run_name].shape=(14042, 7)
GSOFT-code 0 shot: preds_dfs[run_name].shape=(14042, 7)
GSOFT-code 5 shot: preds_dfs[run_name].shape=(14042, 7)


In [7]:
idx = 3

print(preds_dfs['GSOFT-code 0 shot']['text'][idx])
print(preds_dfs['GSOFT-code 0 shot']['model_pred'][idx])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

The following are multiple choice questions (with answers) about abstract_algebra. Output 'A', 'B', 'C', or 'D'. Full answer not needed.<|eot_id|><|start_header_id|>user<|end_header_id|>

Statement 1 | A factor group of a non-Abelian group is non-Abelian. Statement 2 | If K is a normal subgroup of H and H is a normal subgroup of G, then K is a normal subgroup of G.
A. True, True
B. False, False
C. True, False
D. False, True<|eot_id|><|start_header_id|>assistant<|end_header_id|>

B<|eot_id|>
{'generated_text': 'C. True, False'}


In [8]:
tmp_idxs = preds_dfs['GSOFT-code 0 shot']['model_pred'].apply(
    lambda p: re.match('[ABCD]', p['generated_text']) is not None
)

preds_dfs['GSOFT-code 0 shot']['model_pred'][~tmp_idxs].shape

(10,)

In [9]:
def get_category(subject):
    return subcat_to_cat[subject]

# OOD_part = re.compile('The correct answer is [ABCD]')
# OOD_part_len = len('The correct answer is ')

OOD_parts = [
    (re.compile('The correct answer is [ABCD]'), len('The correct answer is ')),
    (re.compile('The answer is [ABCD]'), len('The answer is ')),
    (re.compile('The solution is [ABCD]'), len('The solution is ')),
]

def get_pred(p):
    for OOD_part, OOD_part_len in OOD_parts:
        if OOD_part.search(p):
            return p[OOD_part_len]
    
    return p[0]

def processor(row: pd.DataFrame):
    generated_text = row['model_pred']['generated_text']
    generated_text = generated_text.strip()
    

    # generated_text	subject	pred	true	corr	category
    category = get_category(row['subject'])
    pred = get_pred(generated_text)
    true = chr(ord('A') + row['answer'])
    corr = int(pred == true)

    return {
        'generated_text': generated_text,
        'subject': row['subject'],
        'pred': pred,
        'true': true,
        'corr': corr,
        'category': category
    }

In [10]:
preds_dfs['LoRA-code 0 shot'] = preds_dfs['LoRA-code 0 shot'].apply(processor, axis=1, result_type='expand')
preds_dfs['LoRA-code 5 shot'] = preds_dfs['LoRA-code 5 shot'].apply(processor, axis=1, result_type='expand')

preds_dfs['GSOFT-code 0 shot'] = preds_dfs['GSOFT-code 0 shot'].apply(processor, axis=1, result_type='expand')
preds_dfs['GSOFT-code 5 shot'] = preds_dfs['GSOFT-code 5 shot'].apply(processor, axis=1, result_type='expand')

In [11]:
preds_dfs['LoRA-code 0 shot'].head()

,generated_text,subject,pred,true,corr,category
0,B. 4\nThe degree of a field extension is the d...,abstract_algebra,B,B,1,STEM
1,C. 24\nThe index of an element in a group is t...,abstract_algebra,C,C,1,STEM
2,"D. 0,4\nThe polynomial x^5 + 3x",abstract_algebra,D,D,1,STEM
3,"D. False, True\n\nExplanation: Statement 1 is ...",abstract_algebra,D,B,0,STEM
4,B. 6x^2 + 4x + 6\n\nExplanation,abstract_algebra,B,B,1,STEM


In [12]:
preds_dfs['LoRA-code 5 shot'].head()

,generated_text,subject,pred,true,corr,category
0,B,abstract_algebra,B,B,1,STEM
1,A. 8,abstract_algebra,A,C,0,STEM
2,"D. 0,4",abstract_algebra,D,D,1,STEM
3,D,abstract_algebra,D,B,0,STEM
4,B,abstract_algebra,B,B,1,STEM


In [13]:
preds_dfs['GSOFT-code 0 shot'].head()

,generated_text,subject,pred,true,corr,category
0,C. 2,abstract_algebra,C,B,0,STEM
1,C. 24.,abstract_algebra,C,C,1,STEM
2,"D. 0,4",abstract_algebra,D,D,1,STEM
3,"C. True, False",abstract_algebra,C,B,0,STEM
4,B. 6x^2 + 4x + 6.,abstract_algebra,B,B,1,STEM


In [14]:
preds_dfs['GSOFT-code 5 shot'].head()

,generated_text,subject,pred,true,corr,category
0,B,abstract_algebra,B,B,1,STEM
1,C,abstract_algebra,C,C,1,STEM
2,D,abstract_algebra,D,D,1,STEM
3,D,abstract_algebra,D,B,0,STEM
4,B,abstract_algebra,B,B,1,STEM


In [16]:
preds_dfs['GSOFT-code 0 shot']['pred'].value_counts()

pred
B    5050
C    3563
D    3400
A    2017
H       5
T       2
I       2
0       1
O       1
s       1
Name: count, dtype: int64

In [17]:
acc_by_subjects = pd.DataFrame({
    'subject': list(set.union(
        *map(
            lambda df: set(df['subject']),
            preds_dfs.values()
        )
    ))
})

print(f"Subjects: {acc_by_subjects['subject']}")

for run_name, preds_df in preds_dfs.items():
    acc_df = preds_df[['subject', 'corr']].groupby(['subject'], as_index=False).mean()
    acc_df.rename(columns={'corr': run_name}, inplace=True)

    acc_by_subjects = acc_by_subjects.merge(
        acc_df,
        left_on='subject',
        right_on='subject',
        how='right'
    )

Subjects: 0                  electrical_engineering
1                   high_school_chemistry
2              high_school_macroeconomics
3     high_school_government_and_politics
4                  high_school_statistics
5                     high_school_physics
6                            formal_logic
7                               nutrition
8                               astronomy
9                        medical_genetics
10                 elementary_mathematics
11                       machine_learning
12                      international_law
13                        world_religions
14                              marketing
15                          jurisprudence
16                       security_studies
17                    college_mathematics
18                            human_aging
19                           global_facts
20                       professional_law
21                        business_ethics
22                           econometrics
23                profes

In [18]:
acc_by_subjects

,subject,raw 0 shot,raw 5 shot,LoRA 0 shot,LoRA 5 shot,LoRA-code 0 shot,LoRA-code 5 shot,GSOFT-code 0 shot,GSOFT-code 5 shot
0,abstract_algebra,0.350000,0.310000,0.310000,0.340000,0.350000,0.360000,0.360000,0.340000
1,anatomy,0.674074,0.666667,0.629630,0.614815,0.674074,0.674074,0.718519,0.711111
2,astronomy,0.750000,0.723684,0.717105,0.730263,0.697368,0.730263,0.723684,0.743421
3,business_ethics,0.660000,0.670000,0.610000,0.650000,0.630000,0.660000,0.620000,0.720000
4,clinical_knowledge,0.716981,0.747170,0.743396,0.769811,0.747170,0.754717,0.720755,0.754717
5,college_biology,0.715278,0.756944,0.784722,0.791667,0.708333,0.770833,0.701389,0.756944
6,college_chemistry,0.450000,0.460000,0.480000,0.490000,0.460000,0.490000,0.400000,0.490000
7,college_computer_science,0.470000,0.570000,0.470000,0.540000,0.500000,0.530000,0.400000,0.530000
8,college_mathematics,0.350000,0.350000,0.280000,0.380000,0.390000,0.370000,0.310000,0.380000
9,college_medicine,0.618497,0.641618,0.664740,0.647399,0.578035,0.612717,0.612717,0.618497


In [19]:
acc_by_categories = pd.DataFrame({
    'category': list(set.union(
        *map(
            lambda df: set(df['category']),
            preds_dfs.values()
        )
    ))
})

for run_name, preds_df in preds_dfs.items():
    acc_df = preds_df[['category', 'corr']].groupby(['category'], as_index=False).mean()
    acc_df.rename(columns={'corr': run_name}, inplace=True)

    acc_by_categories = acc_by_categories.merge(
        acc_df,
        left_on='category',
        right_on='category',
        how='right'
    )

In [20]:
acc_by_categories.loc[:, list(filter(
    lambda s: ('5 shot' not in s),
    acc_by_categories.columns
))]

,category,raw 0 shot,LoRA 0 shot,LoRA-code 0 shot,GSOFT-code 0 shot
0,STEM,0.525182,0.540755,0.521206,0.508284
1,humanities,0.527099,0.556217,0.516897,0.525399
2,"other (business, health, misc.)",0.706663,0.709130,0.708205,0.709130
3,social sciences,0.721807,0.746831,0.733507,0.724082


In [22]:
acc_by_categories.loc[:, list(filter(
    lambda s: ('0 shot' not in s),
    acc_by_categories.columns
))]

,category,raw 5 shot,LoRA 5 shot,LoRA-code 5 shot,GSOFT-code 5 shot
0,STEM,0.546720,0.561962,0.560636,0.555003
1,humanities,0.603826,0.591711,0.607226,0.609139
2,"other (business, health, misc.)",0.724244,0.719926,0.731339,0.735349
3,social sciences,0.758531,0.766006,0.760156,0.766331


In [23]:
total_res = pd.DataFrame(columns=['run_name', 'accuracy', 'correctness'])

for run_name, preds_df in preds_dfs.items():
    total_accuracy = preds_df['corr'].mean()
    total_correctness = (preds_df['pred'] != 'I').mean()

    total_res = pd.concat([
        total_res,
        pd.DataFrame({
            'run_name': [run_name],
            'accuracy': [total_accuracy],
            'correctness': [total_correctness],
        })
    ])

C:\Users\Vladimir\AppData\Local\Temp\ipykernel_21356\2066953499.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  total_res = pd.concat([


In [24]:
total_res

,run_name,accuracy,correctness
0,raw 0 shot,0.610810,0.997151
0,raw 5 shot,0.653255,0.999217
0,LoRA 0 shot,0.629967,0.992024
0,LoRA 5 shot,0.653112,1.000000
0,LoRA-code 0 shot,0.609457,0.999929
0,LoRA-code 5 shot,0.659379,1.000000
0,GSOFT-code 0 shot,0.607677,0.999858
0,GSOFT-code 5 shot,0.661088,1.000000
